In [40]:
import fitz
import os
import re
from typing import Union, List

In [41]:
def find_first_long_block(article, min_words=50):
    if len(article) == 0:
        return None
    
    page = article[0]
    blocks = page.get_text("dict")["blocks"]
    
    for i, block in enumerate(blocks):
        if block["type"] != 0:
            continue
        block_text = ""
        for line in block["lines"]:
            for span in line["spans"]:
                block_text += span["text"] + " "
        block_text = block_text.strip()
        if not block_text:
            continue

        word_count = len(block_text.split())
        
        if word_count >= min_words:
            is_likely_heading = (
                len(block_text) < 100 or
                block_text.isupper() or
                block_text[0].isdigit()
            )
            if not is_likely_heading:
                return 0, i, block, "fallback_long_block"
    
    for i, block in enumerate(blocks):
        if block["type"] == 0:
            block_text = ""
            for line in block["lines"]:
                for span in line["spans"]:
                    block_text += span["text"] + " "
            if block_text.strip():
                return 0, i, block, "fallback_first_block"
    
    return None
    

In [59]:
def find_abstract_block(article):
    abst_name_vars = ['abstract', 'abstract.', 'аннотация', 'аннотация.']
    for page_num in range(2):
        page = article[page_num]
        blocks = page.get_text('dict')["blocks"]
        idx = None
        for i, block in enumerate(blocks):
            if block['type'] != 0:
                continue
            block_text = ""
            for line in block['lines']:
                for span in line["spans"]:
                    block_text += span["text"] + " "
            block_text = block_text.strip().lower()

            for var in abst_name_vars:
                if var in block_text:
                    if block_text.startswith(var):
                        return page_num, i, block, "by_keyword"
    
    return find_first_long_block(article)

In [60]:
def clean_abstract_text(text):
    text = " ".join(text.split())
    text = text.replace("-\n", "")
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [97]:
def extract_abstract_text(article, page_num, block_index, block, method):
    page = article[page_num]
    blocks = page.get_text("dict")["blocks"]
    
    abstract_parts = []
    if method == "by_keyword":
        for line in block["lines"]:
            for span in line["spans"]:
                text = span["text"]
                if "abstract" in text.lower():
                    lower_text = text.lower()
                    pos = lower_text.find("abstract")
                    if pos >= 0:
                        after = text[pos + len("abstract"):]
                        after = after.lstrip(": .")
                        print(after)
                        if after:
                            abstract_parts.append(after)
                else:
                    if abstract_parts:
                        abstract_parts.append(text)
        print(abstract_parts)
        if not abstract_parts:
            method = "separate_blocks"

    if method != "by_keyword" or not abstract_parts:
        start_index = block_index + 1
        abstract_font_size = None
        for line in block["lines"]:
            for span in line["spans"]:
                if abstract_font_size is None:
                    abstract_font_size = span["size"]
        for i in range(start_index, len(blocks)):
            curr_block = blocks[i]
            if curr_block["type"] != 0:
                continue
            block_text = ""
            block_font_sizes = []
            for line in curr_block["lines"]:
                for span in line["spans"]:
                    block_text += span["text"] + " "
                    block_font_sizes.append(span["size"])
            
            block_text = block_text.strip()
            if not block_text:
                continue
            avg_font = sum(block_font_sizes) / len(block_font_sizes) if block_font_sizes else 0
            
            is_heading = False
            if len(block_text) < 80:
                if abstract_font_size and avg_font > abstract_font_size * 1.2:
                    is_heading = True
                if block_text[0].isdigit():
                    is_heading = True
                if block_text.isupper():
                    is_heading = True
                heading_words = ['introduction', 'background', 'methods', 'results', 'references']
                if any(word in block_text.lower() for word in heading_words):
                    is_heading = True
            
            if is_heading and abstract_parts:
                print(f"СТОП: найден заголовок")
                print(f"  Текст блока: {block_text[:100]}")
                print(f"  Длина: {len(block_text)}")
                print(f"  avg_font: {avg_font}, abstract_font: {abstract_font_size}")
                break
                
            abstract_parts.append(block_text)
    
    full_abstract = " ".join(abstract_parts)
    
    full_abstract = clean_abstract_text(full_abstract)
    
    return full_abstract

In [98]:
def extract_abstract(
    pdf_path : str
):
    if not os.path.exists(pdf_path): 
        raise FileNotFoundError("Article not found")
    
    article = fitz.open(pdf_path)

    abs_block = find_abstract_block(article)
    if abs_block is None:
        article.close()
        return None
 
    page_num, block_index, block, method = abs_block
    print(method)
    abstract = extract_abstract_text(article, page_num, block_index, block, method)
    article.close()
    return abstract if abstract else None

In [99]:
extract_abstract('testing_data/2603.18162v1.pdf')

by_keyword

[]
СТОП: найден заголовок
  Текст блока: 1
  Длина: 1
  avg_font: 8.966400146484375, abstract_font: 9.962599754333496


'Introduction Let A = { a 0 , a 1 , . . . , a n } be a finite subset of N d for some d ≥ 1 and n ≥ 0 , where a i = ( a i 1 , . . . , a id ) for all i ∈{ 0 , . . . , n } . Given s ∈ N , the s -fold sumset of A , s A , is defined by 0 A := { 0 } and, for s ≥ 1 , s A := { a i 1 + · · · + a i s : 0 ≤ i 1 ≤· · · ≤ i s ≤ n } . Additive combinatorics studies the sumsets of A and their cardinality. Khovanskii proved in [16] that the function N → N defined by s 7→| s A| is asymptotically poly- nomial. Later, Colarte-Gómez, Elias and Miró-Roig in [4] and, independently, Eliahou and Mazumdar in [7], gave new proofs of Khovanskii’s result. In [4], the authors asso- ciate to A a projective toric variety whose Hilbert function coincides with the function s 7→| s A| as follows. Set D := max {| a i | = P d j =1 a ij : i = 0 , . . . , n } , and define A = { a 0 , . . . , a n } ⊂ N d +1 by setting a i = ( D −| a i | , a i 1 , . . . , a id ) ∈ N d +1 for all i . Given an algebraically closed field k , th